In [66]:
import pandas as pd
import json
import os
import re

In [68]:

party_labels = {
    # Democrats (0)
    "Andrew Jackson": 0, "Martin Van Buren": 0, "James K. Polk": 0, "Franklin Pierce": 0, 
    "James Buchanan": 0, "Grover Cleveland": 0, "Woodrow Wilson": 0, "Franklin D. Roosevelt": 0, 
    "Harry S. Truman": 0, "John F. Kennedy": 0, "Lyndon B. Johnson": 0, "Jimmy Carter": 0, 
    "Bill Clinton": 0, "Barack Obama": 0, "Joe Biden": 0, "Kamala Harris": 0,
    
    # Republicans (1)
    "Abraham Lincoln": 1, "Ulysses S. Grant": 1, "Rutherford B. Hayes": 1, "James A. Garfield": 1, 
    "Chester A. Arthur": 1, "Benjamin Harrison": 1, "William McKinley": 1, "Theodore Roosevelt": 1, 
    "William Howard Taft": 1, "Warren G. Harding": 1, "Calvin Coolidge": 1, "Herbert Hoover": 1, 
    "Dwight D. Eisenhower": 1, "Richard Nixon": 1, "Gerald Ford": 1, "Ronald Reagan": 1, 
    "George H.W. Bush": 1, "George W. Bush": 1, "Donald Trump": 1, "Mike Pence": 1
}

def clean_speech_noise(text):
    """Universal text cleaner."""
    text = str(text).replace('<br />', ' ').replace('*', '')
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

master_texts = []
master_labels = []

folder_path = '/Users/karinamehta/Desktop/speeches'
if os.path.exists(folder_path):
    for filename in os.listdir(folder_path):
        if filename.endswith('.json'):
            with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as f:
                data = json.load(f)
                president = data.get('president')
                
                # If they are in our strict dictionary, they are Dem or Rep!
                if president in party_labels:
                    clean_transcript = clean_speech_noise(data.get('transcript'))
                    master_texts.append(clean_transcript)
                    master_labels.append(party_labels[president])
    print("JSONs done")

csv_file = 'us_2020_election_speeches.csv'
if os.path.exists(csv_file):
    df_csv = pd.read_csv(csv_file)
    for _, row in df_csv.iterrows():
        speaker = str(row['speaker']).strip()
        
        if speaker in party_labels:
            clean_transcript = clean_speech_noise(row['text'])
            
            # Remove the first two words (the name/timestamp artifact from this specific dataset)
            words = clean_transcript.split()
            speech_without_name = " ".join(words[2:])
            
            master_texts.append(speech_without_name)
            master_labels.append(party_labels[speaker])
    print("2020 CSV done")


tsv_files = [
    'cleantext_DonaldTrump.tsv', 
    'cleantext_JoeBiden.tsv', 
    'cleantext_KamalaHarris.tsv', 
    'cleantext_MikePence.tsv'
]

for file in tsv_files:
    if os.path.exists(file):
        df_tsv = pd.read_csv(file, sep='\t')
        for _, row in df_tsv.iterrows():
            speaker = str(row['POTUS']).strip()
            
            if speaker in party_labels:
                clean_transcript = clean_speech_noise(row['CleanText'])
                master_texts.append(clean_transcript)
                master_labels.append(party_labels[speaker])
    print("TSVs done")


final_df = pd.DataFrame({'Text': master_texts, 'Label': master_labels})

#shuffling data
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

final_df.to_csv('ultimate_dem_rep_speeches.csv', index=False)

JSONs done
2020 CSV done
TSVs done
TSVs done
TSVs done
TSVs done
Label
0    0.521684
1    0.478316
Name: proportion, dtype: float64
